# Entraînement MKAN — Pipeline complet 

**Flux** :
1. Chargement de `featuresLog.parquet` issu de `MOMTSIM/main.py`
2. Normalisation → fenêtre glissante W × 12 features
3. Découpage temporel train / val / test (section 3.1)
4. Entraînement `MKANScorer` via `mkan_total_loss` (section 4.3.4)
5. Évaluation MCC / AUC-ROC (baseline XGBoost = 0,82)
6. Détection de dérive JS + extension de grille (section 4.4)
7. Élagage + régression symbolique → rapport d'audit COBAC (section 4.4.7)

In [1]:
import sys, os

# Le notebook est dans MKAN/ → le parent (..) est Modelisation/
ROOT        = os.path.abspath("..")
MOMTSIM_DIR = os.path.join(ROOT, "MOMTSIM")
for p in [ROOT, MOMTSIM_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Package MKAN (MKAN/__init__.py)
from MKAN import (
    MKANScorer, mkan_total_loss,
    js_divergence, detect_drift_region,
    extract_full_model_report,
)

print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())
print("sys.path[0:2] :", sys.path[:2])

PyTorch 2.5.1+cpu | CUDA False
sys.path[0:2] : ['m:\\Ecole\\Mémoire\\Modelisation\\MOMTSIM', 'm:\\Ecole\\Mémoire\\Modelisation']


## 1. Configuration

In [2]:
# ── Chemins ────────────────────────────────────────────────────────────────
FEATURES_FILE = os.path.join("..", "MOMTSIM", "config", "featuresLog.parquet")  # sortie de main.py
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Features (eq. 3.8–3.19, section 3.2.6) ────────────────────────────────
FEATURE_COLS = [
    "delta_B_orig",          # eq. 3.8  — ΔB_orig
    "delta_B_dest",          # eq. 3.9  — ΔB_dest
    "r1",                    # eq. 3.10 — montant / ancien solde orig
    "r2",                    # eq. 3.11 — montant / nouveau solde orig
    "flag_anomalie",         # eq. 3.12 — Flag_anomalie ∈ {0,1}
    "delta_commission",      # eq. 3.13 — δ_commission (smurfing)
    "var_agent_split",       # eq. 3.14 — Var_agent (split deposit)
    "rho_rupture",           # eq. 3.15 — ρ_rupture (fake credentials)
    "rho_refund",            # eq. 3.16 — ρ_refund (refund fraud)
    "v1h",                   # eq. 3.17 — V_1h (vélocité)
    "flag_nuit",             # eq. 3.18 — Flag_nuit ∈ {0,1}
    "rho_nouveau",           # eq. 3.19 — ρ_nouveau (destinataires inconnus)
]
TARGET_COL   = "isFraud"
ACCOUNT_COL  = "nameOrig"
TIME_COL     = "step"

# ── Architecture MKAN (section 4.2) ───────────────────────────────────────
INPUT_SIZE  = len(FEATURE_COLS)   # 12
HIDDEN_SIZE = 32                  # dimension h_t / c_t
W           = 10                  # fenêtre glissante (pas de temps)
M_GAUSS     = 8                   # centres gaussiens par arête (section 4.2.2)
K_FOURIER   = 2                   # harmoniques Fourier    (section 4.2.2)

# ── Entraînement (section 4.3.4) ──────────────────────────────────────────
BATCH_SIZE  = 256
LR          = 1e-3
N_EPOCHS    = 40
LAM         = 1e-2    # force de régularisation globale
MU1         = 1.0     # poids norme L1
MU2         = 0.5     # poids entropie

# ── Découpage temporel ─────────────────────────────────────────────────────
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15   # (test = 1 - 0.70 - 0.15 = 0.15)

# ── Seuil de drift JS (section 4.4.2, eq. 4.19) ───────────────────────────
JS_THRESHOLD = 0.05

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositif : {DEVICE}")

Dispositif : cpu


## 2. Chargement et exploration des données

In [3]:
df = pd.read_parquet(FEATURES_FILE)

# Remplissage des NaN avant toute normalisation
# delta_commission et var_agent_split sont NaN pour les tx où la feature ne s'applique pas :
# → 0 est la valeur neutre sémantiquement correcte (aucune activité de mule / aucun split)
nan_counts = df[FEATURE_COLS].isna().sum()
if nan_counts.any():
    print("NaN détectés (remplacement par 0) :")
    print(nan_counts[nan_counts > 0])
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)

print(f"\nTransactions totales : {len(df):,}")
print(f"Taux de fraude global : {df[TARGET_COL].mean():.3f}")
print(f"Steps : {df[TIME_COL].min()} → {df[TIME_COL].max()}")
print(f"Comptes uniques (nameOrig) : {df[ACCOUNT_COL].nunique():,}")
if df['fraudScenario'].notna().any():
    print("\nRépartition par scénario :")
    print(df.loc[df[TARGET_COL], 'fraudScenario'].value_counts())

NaN détectés (remplacement par 0) :
delta_commission    21452
var_agent_split      8767
dtype: int64

Transactions totales : 21,452
Taux de fraude global : 0.081
Steps : 1 → 719
Comptes uniques (nameOrig) : 2,228

Répartition par scénario :
fraudScenario
SPLIT_DEP    1250
REFUND        485
Name: count, dtype: int64


In [4]:
# Statistiques des 12 features
df[FEATURE_COLS + [TARGET_COL]].describe().round(4)

,delta_B_orig,delta_B_dest,r1,r2,delta_commission,var_agent_split,rho_rupture,rho_refund,v1h,rho_nouveau
count,21452.0000,2.145200e+04,2.145200e+04,21452.0000,21452.0,2.145200e+04,2.145200e+04,2.145200e+04,21452.0000,21452.0000
mean,23589.6175,4.404165e+04,8.306091e+02,-0.4459,0.0,1.146088e+08,1.252002e+10,4.311952e+04,1.3164,0.3647
std,44921.7462,1.007453e+05,5.044890e+04,57.5663,0.0,7.426066e+08,3.896807e+10,3.762762e+05,0.8070,0.2489
min,-119253.0312,-2.684965e+05,-9.717904e+03,-8046.3821,0.0,0.000000e+00,0.000000e+00,0.000000e+00,1.0000,0.0455
25%,1.0000,1.000000e+00,-3.120000e-02,-0.0964,0.0,0.000000e+00,0.000000e+00,0.000000e+00,1.0000,0.2000
50%,2453.3047,2.810516e+03,-0.000000e+00,-0.0025,0.0,0.000000e+00,6.740000e-02,0.000000e+00,1.0000,0.2857
75%,22736.5586,2.774859e+04,-0.000000e+00,-0.0000,0.0,0.000000e+00,7.342000e-01,0.000000e+00,1.0000,0.5000
max,412843.3750,1.123131e+06,7.059850e+06,2380.4868,0.0,2.237515e+10,3.535766e+11,4.000000e+06,6.0000,1.0000


## 3. Normalisation standardisée (section 4.1.1, eq. 4.1)

$$\tilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j + \varepsilon}$$

Paramètres ajustés **uniquement sur le train** pour éviter la fuite de données.

In [5]:
step_max    = df[TIME_COL].max()
train_end   = int(step_max * TRAIN_FRAC)
val_end     = int(step_max * (TRAIN_FRAC + VAL_FRAC))

df_train = df[df[TIME_COL] <= train_end].copy()
df_val   = df[(df[TIME_COL] > train_end) & (df[TIME_COL] <= val_end)].copy()
df_test  = df[df[TIME_COL] > val_end].copy()

print(f"Train : {len(df_train):,} tx  ({df_train[TARGET_COL].mean():.3f} fraude)")
print(f"Val   : {len(df_val):,} tx  ({df_val[TARGET_COL].mean():.3f} fraude)")
print(f"Test  : {len(df_test):,} tx  ({df_test[TARGET_COL].mean():.3f} fraude)")

# Normalisation sur le train uniquement
mu_train  = df_train[FEATURE_COLS].mean()
std_train = df_train[FEATURE_COLS].std().clip(lower=1e-6)

for d in [df_train, df_val, df_test]:
    d[FEATURE_COLS] = (d[FEATURE_COLS] - mu_train) / std_train

print("\nNormalisation appliquée.")

Train : 20,544 tx  (0.059 fraude)
Val   : 475 tx  (0.543 fraude)
Test  : 433 tx  (0.600 fraude)

Normalisation appliquée.


## 4. Construction des fenêtres glissantes (eq. 4.16)

Pour chaque compte `nameOrig`, on crée toutes les fenêtres de $W$ transactions consécutives.
Le label d'une fenêtre est l'étiquette `isFraud` de la **dernière** transaction de la fenêtre.

In [6]:
def make_windows(df_split: pd.DataFrame, W: int):
    """
    Fenêtres glissantes par compte (eq. 4.16) : (batch, W, 12) → label de la dernière tx.
    Comptes avec < W transactions sont ignorés (fenêtre impossible).
    """
    X_list, y_list = [], []
    feat = df_split[FEATURE_COLS].values.astype(np.float32)
    targ = df_split[TARGET_COL].values.astype(np.float32)
    acct = df_split[ACCOUNT_COL].values
    step = df_split[TIME_COL].values

    _, unique_starts = np.unique(acct, return_index=True)
    unique_ends = np.append(unique_starts[1:], len(acct))

    for start, end in zip(unique_starts, unique_ends):
        n = end - start
        if n < W:
            continue
        # Tri par step au sein du compte (déjà trié globalement, mais sécurité)
        order = np.argsort(step[start:end])
        Xacc = feat[start:end][order]
        yacc = targ[start:end][order]
        for i in range(n - W + 1):
            X_list.append(Xacc[i:i+W])
            y_list.append(yacc[i+W-1])

    X = np.array(X_list, dtype=np.float32)   # (N_windows, W, 12)
    y = np.array(y_list, dtype=np.float32)   # (N_windows,)
    return X, y


# Trier par compte puis par step avant découpage
for d in [df_train, df_val, df_test]:
    d.sort_values([ACCOUNT_COL, TIME_COL], inplace=True)
    d.reset_index(drop=True, inplace=True)

X_train, y_train = make_windows(df_train, W)
X_val,   y_val   = make_windows(df_val,   W)
X_test,  y_test  = make_windows(df_test,  W)

print(f"Fenêtres train : {len(X_train):,}  (fraude : {y_train.mean():.3f})")
print(f"Fenêtres val   : {len(X_val):,}  (fraude : {y_val.mean():.3f})")
print(f"Fenêtres test  : {len(X_test):,}  (fraude : {y_test.mean():.3f})")

Fenêtres train : 3,450  (fraude : 0.059)
Fenêtres val   : 51  (fraude : 0.667)
Fenêtres test  : 48  (fraude : 0.604)


In [7]:
def to_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = to_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

## 5. Instanciation du modèle MKAN (section 4.2.4, eq. 4.16)

In [8]:
model = MKANScorer(
    input_size  = INPUT_SIZE,    # 12 features
    hidden_size = HIDDEN_SIZE,   # 32 dimensions
    M           = M_GAUSS,       # 8 centres gaussiens
    K           = K_FOURIER,     # 2 harmoniques Fourier
    domain      = 1.0,           # domaine normalisé [-1, 1]
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {n_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Paramètres entraînables : 67,617


## 6. Boucle d'entraînement (ℓ_total, section 4.3.4)

$$\ell_{total} = \ell_{pred} + \lambda\left(\mu_1 \sum_l |\Phi_l|_1 + \mu_2 \sum_l S(\Phi_l)\right)$$

In [9]:
@torch.no_grad()
def evaluate(model, loader):
    """
    Métriques de détection de fraude — implémentation pure numpy (section 3.4).

    MCC  (Matthews Correlation Coefficient, eq. 3.1) :
        MCC = (TP·TN - FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN))

    AUC-ROC par règle trapézoïdale (eq. 3.2) :
        AUC = ∫ TPR d(FPR)  via np.trapz sur la courbe ROC triée par score décroissant

    PR-AUC par règle trapézoïdale (eq. 3.6) :
        PR-AUC = ∫ Précision d(Rappel)  via np.trapz sur la courbe PR

    Score de Brier (eq. 3.7) :
        Brier = (1/N) Σ (ŷᵢ - yᵢ)²

    Précision = TP / (TP + FP)   (eq. 3.4)
    Rappel    = TP / (TP + FN)   (eq. 3.4)
    F1        = 2 · Précision · Rappel / (Précision + Rappel)   (eq. 3.5)
    """
    model.eval()
    all_scores, all_labels = [], []
    for X_batch, y_batch in loader:
        all_scores.append(model(X_batch.to(DEVICE)).cpu().numpy())
        all_labels.append(y_batch.numpy())

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (scores >= 0.5).astype(int)

    # Éléments de la matrice de confusion
    tp = int(((preds == 1) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())

    # MCC (eq. 3.1)
    num = tp * tn - fp * fn
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = float(num / (den + 1e-12))

    # AUC-ROC trapézoïdale (eq. 3.2)
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos > 0 and n_neg > 0:
        sidx   = np.argsort(-scores)
        labs_s = labels[sidx].astype(float)
        tpr    = np.concatenate([[0.0], np.cumsum(labs_s)       / n_pos])
        fpr    = np.concatenate([[0.0], np.cumsum(1 - labs_s)   / n_neg])
        auc    = float(abs(np.trapz(tpr, fpr)))

        # PR-AUC trapézoïdale (eq. 3.6)
        cum_pos    = np.cumsum(labs_s)
        cum_n      = np.arange(1, len(labs_s) + 1, dtype=float)
        prec_curve = np.concatenate([[1.0], cum_pos / cum_n])
        rec_curve  = np.concatenate([[0.0], cum_pos / n_pos])
        pr_auc     = float(abs(np.trapz(prec_curve, rec_curve)))
    else:
        auc    = float("nan")
        pr_auc = float("nan")

    # Précision, Rappel, F1 (eq. 3.4–3.5)
    prec = float(tp / (tp + fp + 1e-12))
    rec  = float(tp / (tp + fn + 1e-12))
    f1   = float(2 * prec * rec / (prec + rec + 1e-12))

    # Score de Brier (eq. 3.7)
    brier = float(np.mean((scores - labels.astype(float)) ** 2))

    return dict(mcc=mcc, auc=auc, pr_auc=pr_auc, brier=brier,
                precision=prec, recall=rec, f1=f1)


In [10]:
history = {"epoch": [], "loss": [], "pred_loss": [], "reg": [],
           "l1": [], "entropy": [],
           "val_mcc": [], "val_auc": [], "val_prauc": [], "val_brier": []}

best_val_mcc = -1.0
best_epoch   = 0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = epoch_pred = epoch_l1 = epoch_ent = 0.0
    n_batches  = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()

        loss, pred_loss, reg_l1, reg_entropy = mkan_total_loss(
            model, X_batch, y_batch, lam=LAM, mu1=MU1, mu2=MU2)

        loss.backward()
        # Clipping des gradients (stabilisation des splines KAN, section 2.3.2)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_pred += pred_loss.item()
        epoch_l1   += reg_l1.item()
        epoch_ent  += reg_entropy.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    avg_pred = epoch_pred / n_batches
    avg_l1   = epoch_l1  / n_batches
    avg_ent  = epoch_ent / n_batches

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["pred_loss"].append(avg_pred)
    history["reg"].append(avg_loss - avg_pred)
    history["l1"].append(avg_l1)
    history["entropy"].append(avg_ent)

    val_metrics = evaluate(model, val_loader)
    history["val_mcc"].append(val_metrics["mcc"])
    history["val_auc"].append(val_metrics["auc"])
    history["val_prauc"].append(val_metrics["pr_auc"])
    history["val_brier"].append(val_metrics["brier"])

    if val_metrics["mcc"] > best_val_mcc:
        best_val_mcc = val_metrics["mcc"]
        best_epoch   = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_mkan.pt"))

    if epoch % 5 == 0 or epoch == 1:
        print(f"[{epoch:3d}/{N_EPOCHS}] "
              f"loss={avg_loss:.4f}  "
              f"pred={avg_pred:.4f}  "
              f"L1={avg_l1:.4f}  "
              f"ent={avg_ent:.4f}  "
              f"val_MCC={val_metrics['mcc']:.4f}  "
              f"val_PRAUC={val_metrics['pr_auc']:.4f}  "
              f"val_Brier={val_metrics['brier']:.4f}")

print(f"\nMeilleur val MCC = {best_val_mcc:.4f} (epoch {best_epoch})")


[  1/40] loss=73.8216  pred=0.6886  L1=7175.3246  ent=275.9554  val_MCC=0.0000  val_PRAUC=0.7600  val_Brier=0.2454
[  5/40] loss=13.7856  pred=0.6880  L1=1184.3274  ent=250.8512  val_MCC=0.0000  val_PRAUC=0.5765  val_Brier=0.2491
[ 10/40] loss=6.9737  pred=0.6892  L1=503.8118  ent=249.2772  val_MCC=0.0000  val_PRAUC=0.5790  val_Brier=0.2466
[ 15/40] loss=5.4252  pred=0.6893  L1=347.3020  ent=252.5677  val_MCC=0.0000  val_PRAUC=0.6959  val_Brier=0.2477
[ 20/40] loss=4.4041  pred=0.6887  L1=244.0205  ent=255.0327  val_MCC=0.0000  val_PRAUC=0.7609  val_Brier=0.2487
[ 25/40] loss=3.7427  pred=0.6876  L1=177.1000  ent=256.8271  val_MCC=0.0000  val_PRAUC=0.7723  val_Brier=0.2490
[ 30/40] loss=3.3500  pred=0.6860  L1=136.4006  ent=260.0162  val_MCC=0.1768  val_PRAUC=0.7028  val_Brier=0.2501
[ 35/40] loss=3.1413  pred=0.6840  L1=114.2432  ent=262.9588  val_MCC=0.0000  val_PRAUC=0.7428  val_Brier=0.2486
[ 40/40] loss=3.0650  pred=0.6814  L1=105.4936  ent=265.7374  val_MCC=0.0000  val_PRAUC=0.79

## 7. Visualisation de la convergence

In [11]:
# Reload MKAN pour inclure MKANVisualizer si le kernel avait l'ancienne version
import importlib, sys as _sys
for _k in [_k for _k in _sys.modules if _k.startswith('MKAN')]:
    del _sys.modules[_k]
from MKAN import MKANVisualizer

# Instanciation unique du visualiseur (reutilise dans toutes les cellules suivantes)
viz = MKANVisualizer(
    model         = model,
    history       = history,
    feature_names = FEATURE_COLS,
    hidden_size   = HIDDEN_SIZE,
)
print(f'MKANVisualizer pret — {len(viz.concat_labels)} entrees: {viz.concat_labels[:3]}...')

# Tableau de bord complet : perte, regularisation, metriques (section 4.4.6)
fig_dashboard = viz.plot_training_dashboard()
fig_dashboard.write_html(os.path.join(CHECKPOINT_DIR, 'dashboard.html'))
fig_dashboard.show()
print(f'Dashboard sauvegarde : {CHECKPOINT_DIR}/dashboard.html')


MKANVisualizer pret — 44 entrees: ['h[0]', 'h[1]', 'h[2]']...


Dashboard sauvegarde : checkpoints/dashboard.html


## 8. Évaluation finale sur le test

Charge le meilleur checkpoint (val MCC maximal).

In [12]:
# Rechargement du meilleur checkpoint
model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, "best_mkan.pt"), map_location=DEVICE))

test_metrics = evaluate(model, test_loader)
print("=== Résultats TEST ===")
for k, v in test_metrics.items():
    print(f"  {k:12s} : {v:.4f}")

print(f"\nBaseline XGBoost (Azamuke et al. 2025) : MCC = 0.82, AUC = 0.97")
delta_mcc = test_metrics['mcc'] - 0.82
print(f"Écart MKAN / XGBoost   : ΔMCC = {delta_mcc:+.4f}")

=== Résultats TEST ===
  mcc          : -0.1656
  auc          : 0.4083
  pr_auc       : 0.6038
  brier        : 0.2503
  precision    : 0.5357
  recall       : 0.5172
  f1           : 0.5263

Baseline XGBoost (Azamuke et al. 2025) : MCC = 0.82, AUC = 0.97
Écart MKAN / XGBoost   : ΔMCC = -0.9856


C:\Users\Miguel\AppData\Local\Temp\ipykernel_27372\3318099978.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(CHECKPOINT_DIR, "best_mkan.pt"), ma

In [13]:
# Collecte des prédictions sur le jeu de test
model.eval()
all_scores_t, all_labels_t = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        all_scores_t.append(model(X_batch.to(DEVICE)).cpu().numpy())
        all_labels_t.append(y_batch.numpy())

scores_t = np.concatenate(all_scores_t)
labels_t = np.concatenate(all_labels_t).astype(int)
preds_t  = (scores_t >= 0.5).astype(int)

# Matrice de confusion (pure numpy)
tn = int(((preds_t == 0) & (labels_t == 0)).sum())
fp = int(((preds_t == 1) & (labels_t == 0)).sum())
fn = int(((preds_t == 0) & (labels_t == 1)).sum())
tp = int(((preds_t == 1) & (labels_t == 1)).sum())

cm_values   = [[tn, fp], [fn, tp]]
axis_labels = ['Légitime', 'Fraude']
annot_text  = [[str(v) for v in row] for row in cm_values]

fig_cm = go.Figure(go.Heatmap(
    z=cm_values,
    x=axis_labels,
    y=axis_labels,
    colorscale='Blues',
    text=annot_text,
    texttemplate='%{text}',
    showscale=True,
))
fig_cm.update_layout(
    title='Matrice de confusion — MKAN (test)',
    xaxis_title='Prédit',
    yaxis_title='Réel',
    width=420, height=370,
)
fig_cm.write_html(os.path.join(CHECKPOINT_DIR, 'confusion_matrix.html'))
fig_cm.show()
print(f'VN={tn}  FP={fp}  FN={fn}  VP={tp}')

VN=6  FP=13  FN=14  VP=15


## 9. Détection de dérive JS (section 4.4, eq. 4.19)

Compare la distribution des scores sur le **val** (référence) avec celle sur le **test** (déploiement simulé).

$$JS(P \| Q) = \frac{1}{2}\left[KL(P \| M) + KL(Q \| M)\right], \quad M = \frac{P+Q}{2}$$

In [14]:
# Pool de reference (val) et pool de deploiement (test) — derniere tx de chaque fenetre
X_val_t  = torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

pool_ref = X_val_t[:, -1, :]    # (N_val,  12)
pool_new = X_test_t[:, -1, :]   # (N_test, 12)

# Divergence JS par feature (eq. 4.19)
# js_divergence attend des histogrammes de MEME longueur — on discretise d'abord
# les echantillons bruts avec des bins communs sur [-domain, domain]
DRIFT_BINS   = 20
DRIFT_DOMAIN = 3.0    # couvre [-3sigma, +3sigma] apres normalisation
bin_edges = np.linspace(-DRIFT_DOMAIN, DRIFT_DOMAIN, DRIFT_BINS + 1)

js_scores = []
for j, fname in enumerate(FEATURE_COLS):
    ref_j = pool_ref[:, j].cpu().numpy()
    new_j = pool_new[:, j].cpu().numpy()
    hist_ref, _ = np.histogram(ref_j, bins=bin_edges)
    hist_new, _ = np.histogram(new_j, bins=bin_edges)
    js = js_divergence(hist_ref.astype(float), hist_new.astype(float))
    js_scores.append(js)
    flag = 'DERIVE' if js > JS_THRESHOLD else 'OK'
    print(f'  {fname:25s}  JS={js:.4f}  {flag}')

drift_features = [FEATURE_COLS[j] for j, js in enumerate(js_scores) if js > JS_THRESHOLD]
print(f'\nFeatures en derive (JS > {JS_THRESHOLD}) : {drift_features}')


  delta_B_orig               JS=0.0147  OK
  delta_B_dest               JS=0.0147  OK
  r1                         JS=-0.0000  OK
  r2                         JS=-0.0000  OK
  flag_anomalie              JS=-0.0000  OK
  delta_commission           JS=-0.0000  OK
  var_agent_split            JS=-0.0000  OK
  rho_rupture                JS=-0.0000  OK
  rho_refund                 JS=-0.0000  OK
  v1h                        JS=0.0153  OK
  flag_nuit                  JS=0.0293  OK
  rho_nouveau                JS=0.1045  DERIVE

Features en derive (JS > 0.05) : ['rho_nouveau']


In [18]:
# Extension de grille (eq. 4.17–4.18) sur les features en dérive
# detect_drift_region retourne (region, js_val) où region = (xl, xr) ou None
if drift_features:
    print('Extension de grille sur les arêtes des portes MKAN...')
    gates_list    = [model.cell.forget_gate, model.cell.input_gate,
                     model.cell.candidate_gate, model.cell.output_gate]
    N_NEW_CENTERS = 4   # centres gaussiens insérés dans la région de dérive

    extended = False
    for fname in drift_features:
        feat_idx = FEATURE_COLS.index(fname)
        region, js_val = detect_drift_region(
            pool_ref[:, feat_idx].cpu().numpy(),
            pool_new[:, feat_idx].cpu().numpy(),
        )
        print(f'  {fname}: JS={js_val:.4f}, région={region}')

        if region is not None:
            for gate in gates_list:
                new_M = gate.extend_grid(region, N_NEW_CENTERS)
            extended = True
            print(f'    -> Grille étendue à M={new_M} centres')

    if extended:
        # Reconstruction de l'optimiseur après extension (nn.Parameter remplacé)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
        print(f'\nOptimiseur reconstruit (lr={LR * 0.1:.1e})')
else:
    print('Aucune dérive détectée — extension de grille non nécessaire.')

Extension de grille sur les arêtes des portes MKAN...
  rho_nouveau: JS=0.1045, région=(-0.3999999999999999, 0.0)
    -> Grille étendue à M=16 centres

Optimiseur reconstruit (lr=1.0e-04)


## 10. Rapport d'audit COBAC — Élagage + Régression symbolique (section 4.4.7)

In [19]:
# Construction du pool d'audit : les portes T-KAN prennent [h_{t-1}, x_t] en entrée
# concat_size = hidden_size + input_size = 32 + 12 = 44
# On collecte les entrées réelles en déroulant le modèle sur les W-1 premiers pas.
model.eval()
N_AUDIT     = min(500, len(X_train))
audit_X     = torch.tensor(X_train[:N_AUDIT], dtype=torch.float32).to(DEVICE)
concat_list = []

with torch.no_grad():
    for b_start in range(0, N_AUDIT, 64):
        xb    = audit_X[b_start:b_start + 64]
        batch = xb.shape[0]
        h_t   = torch.zeros(batch, HIDDEN_SIZE, device=DEVICE)
        c_t   = torch.zeros(batch, HIDDEN_SIZE, device=DEVICE)
        for t in range(W - 1):
            h_t, c_t = model.cell(xb[:, t, :], h_t, c_t)
        # Entrée de la porte au dernier pas : [h_{W-2}, x_{W-1}]
        concat_list.append(torch.cat([h_t, xb[:, W - 1, :]], dim=1).cpu())

x_pool_audit = torch.cat(concat_list, dim=0).to(DEVICE)  # (N_AUDIT, 44)
print(f'Pool d audit : {x_pool_audit.shape}  '
      f'(attendu : [{N_AUDIT}, {HIDDEN_SIZE + INPUT_SIZE}])')

Pool d audit : torch.Size([500, 44])  (attendu : [500, 44])


In [20]:
# Élagage + régression symbolique sur les 4 portes T-KAN (section 4.4.7)
# x_pool_audit : (N, hidden_size + input_size) — entrées réelles des portes
report = extract_full_model_report(
    model,
    x_pool        = x_pool_audit,
    feature_names = FEATURE_COLS,
    theta         = 1e-2,
    r2_threshold  = 0.99,
)

# Statistiques globales
n_active   = sum(len(edges) for edges in report.values())
n_symbolic = sum(1 for edges in report.values()
                 for e in edges if e['symbolifiable'])

print('=== Rapport d audit COBAC ===')
print(f'Arêtes actives   (|φ|₁ > θ=0.01)  : {n_active}')
print(f'Arêtes symbolifiables (R² ≥ 0.99) : {n_symbolic} / {n_active}')
print()

for gate_name, edges in report.items():
    if not edges:
        print(f'Porte {gate_name.upper():10s} — aucune arête active')
        continue
    print(f'Porte {gate_name.upper():10s} ({len(edges)} arêtes actives) :')
    for edge in edges[:5]:   # top 5 par importance L1
        symb = 'OK ' if edge['symbolifiable'] else '~  '
        print(f"  [{symb}] {edge['input']:15s} -> {edge['output']:12s}  "
              f"L1={edge['l1_importance']:.4f}  "
              f"R2={edge['r2']:.4f}  "
              f"{edge['formula']}")
    print()

=== Rapport d audit COBAC ===
Arêtes actives   (|φ|₁ > θ=0.01)  : 32
Arêtes symbolifiables (R² ≥ 0.99) : 0 / 32

Porte FORGET     (8 arêtes actives) :
  [~  ] h[26]           -> h_out[31]     L1=1.6160  R2=0.7597  non-symbolifiable (numerique)
  [~  ] h[6]            -> h_out[26]     L1=1.5682  R2=0.6208  non-symbolifiable (numerique)
  [~  ] delta_commission -> h_out[6]      L1=1.5660  R2=0.8564  non-symbolifiable (numerique)
  [~  ] h[18]           -> h_out[13]     L1=1.5157  R2=0.7508  non-symbolifiable (numerique)
  [~  ] h[26]           -> h_out[3]      L1=1.4938  R2=0.7227  non-symbolifiable (numerique)

Porte INPUT      (8 arêtes actives) :
  [~  ] h[11]           -> h_out[5]      L1=2.2253  R2=0.8307  non-symbolifiable (numerique)
  [~  ] h[31]           -> h_out[5]      L1=2.0083  R2=0.8650  non-symbolifiable (numerique)
  [~  ] h[6]            -> h_out[23]     L1=1.9533  R2=0.8007  non-symbolifiable (numerique)
  [~  ] flag_anomalie   -> h_out[3]      L1=1.6077  R2=0.8067  no

In [21]:
import json

torch.save({
    'model_state': model.state_dict(),
    'config': {
        'input_size':   INPUT_SIZE,
        'hidden_size':  HIDDEN_SIZE,
        'W':            W,
        'M':            M_GAUSS,
        'K':            K_FOURIER,
        'feature_cols': FEATURE_COLS,
        'mu_train':     mu_train.to_dict(),
        'std_train':    std_train.to_dict(),
    },
    'test_metrics':  test_metrics,
    'best_val_mcc':  best_val_mcc,
}, os.path.join(CHECKPOINT_DIR, 'mkan_final.pt'))

with open(os.path.join(CHECKPOINT_DIR, 'audit_report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print('Sauvegardé :')
print(f'  {CHECKPOINT_DIR}/mkan_final.pt         — modèle + config + métriques')
print(f'  {CHECKPOINT_DIR}/audit_report.json      — rapport d audit COBAC')
print(f'  {CHECKPOINT_DIR}/training_curves.html   — courbes Plotly interactives')
print(f'  {CHECKPOINT_DIR}/confusion_matrix.html  — matrice de confusion Plotly')

Sauvegardé :
  checkpoints/mkan_final.pt         — modèle + config + métriques
  checkpoints/audit_report.json      — rapport d audit COBAC
  checkpoints/training_curves.html   — courbes Plotly interactives
  checkpoints/confusion_matrix.html  — matrice de confusion Plotly


## 11. Visualisations structurelles des aretes T-KAN (MKANVisualizer)

`viz` a ete instancie a la section 7. On utilise ici `x_pool_audit` (section 10) pour les methodes necessitant les entrees reelles des portes.

- `plot_edge_heatmap` — heatmap L1 par porte (eq. 2.19)
- `plot_edge_functions` — courbes phi_ij(x) avec decomposition Gaussienne + Fourier (section 4.2.2)
- `plot_pruning_summary` — bilan de sparsification par porte (eq. 4.28)

In [22]:
# Courbes de perte decomposees et metriques individuelles
for title, method in [
    ('loss_curves',     viz.plot_loss_curves),
    ('regularization',  viz.plot_regularization_detail),
    ('val_metrics',     viz.plot_metrics),
]:
    fig = method()
    fig.write_html(os.path.join(CHECKPOINT_DIR, f'{title}.html'))
    fig.show()


In [23]:
# Heatmap L1 des arêtes par porte T-KAN (normes eq. 2.19)
for gate_name in ['forget', 'input', 'candidate', 'output']:
    fig_hm = viz.plot_edge_heatmap(gate_name, x_pool_audit)
    fig_hm.write_html(os.path.join(CHECKPOINT_DIR, f'heatmap_{gate_name}.html'))
    fig_hm.show()
    print(f"Heatmap {gate_name} sauvegardée.")


Heatmap forget sauvegardée.


Heatmap input sauvegardée.


Heatmap candidate sauvegardée.


Heatmap output sauvegardée.


In [24]:
# Courbes φ_ij(x) avec décomposition Gaussienne + Fourier (section 4.2.2)
for gate_name in ['forget', 'input', 'candidate', 'output']:
    try:
        fig_fn = viz.plot_edge_functions(gate_name, x_pool_audit, theta=1e-2, top_k=6)
        fig_fn.write_html(os.path.join(CHECKPOINT_DIR, f'edge_functions_{gate_name}.html'))
        fig_fn.show()
    except ValueError as e:
        print(f"Porte {gate_name} ignorée : {e}")


In [25]:
# Bilan de sparsification : arêtes totales vs survivantes par porte (eq. 4.28)
fig_prune = viz.plot_pruning_summary(x_pool_audit, theta=1e-2)
fig_prune.write_html(os.path.join(CHECKPOINT_DIR, 'pruning_summary.html'))
fig_prune.show()
print(f"Bilan élagage sauvegardé : {CHECKPOINT_DIR}/pruning_summary.html")


Bilan élagage sauvegardé : checkpoints/pruning_summary.html
